# **Práctica Clustering**


## **Clustering**

*Clustering
El un método de *machine learning*, **no supervisados**, cuyo principal objetivo encontrar grupos de datos con patrones similares. Algunos de esos  
algoritmos son [**kmeans**](https://towardsdatascience.com/k-means-clustering-algorithm-applications-evaluation-methods-and-drawbacks-aa03e644b48a), [**mean sheaf**](https://www.geeksforgeeks.org/ml-mean-shift-clustering/) y [**DBScan**](https://www.kdnuggets.com/2020/04/dbscan-clustering-algorithm-machine-learning.html)  


## **Segmentación de imágenes médicas mediante técnicas de clustering**


Para esta práctica vuestro grupo utilizará la siguiente 
[base de datos](https://drive.google.com/file/d/1HuOf5CWxZRsWr-37TIxP5msAqmgsYE4o/view?usp=sharing). Esta base de datos contiene imágenes BMP de tomografías de cerebro.

Tenga en cuanta que as imágenes, a color, son cubos de m x n x k , donde m es el número de filas, n el número de columnas y K=3, en caso de imágenes a color, es el número de canales de la imagen.  Por otro lado, cada pixel, se representa, generalmente como un vector de tamaño k, donde k es el número canales. Luego, podemos pensar en una imagen como un conjunto de puntos en un espacio k  dimensional y por lo tanto, podemos agrupar aquellos pixels cuyos vectores sean lo más similares entre sí, esto aplicando cualquiera de los métodos vistos en clase. 


## **Actividades**

1. Utilizar el algoritmo kmeans, DBscan y meanshift para generar cluster con las imágenes de cerebros.


In [ ]:
%matplotlib widget

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.spatial import distance
from operator import itemgetter
from sklearn.neighbors import NearestNeighbors, KDTree
import time
import imageio
import os

In [ ]:
IMG_PATH = './datos_cerebros/'

In [ ]:
img = Image.open('./datos_cerebros/paciente 1/1/1.BMP')

In [ ]:
mat = np.asarray(img)
# print(mat.shape)
# print(mat[0].shape)
# print(mat[0][0])
mat

In [ ]:
# https://www.geeksforgeeks.org/calculate-the-euclidean-distance-using-numpy/
def euclidean(x1, x2):
    return distance.euclidean(x1, x2)

def chebychev(x1, x2):
    return distance.chebyshev(x1, x2)

def manhattan(x1, x2):
    return distance.cityblock(x1, x2)

In [ ]:
# https://neptune.ai/blog/k-means-clustering
def rand_centroids(data, k, distance):
    r, c = data.shape
    c_idx = np.random.choice(r, k)
    return data.iloc[c_idx].to_numpy()

def plus_plus(data, k, distance):
    centroids = [ data.sample(n=1).to_numpy().tolist()[0] ]
    for _ in range(k-1):
        new_c = ([],-1)         # centroid, distance to c
        for i in range(len(data)):
            x_i = data.iloc[i].to_numpy().tolist()
            if x_i not in centroids:
                d = 0
                for c in centroids:
                    d += pow(distance(c, x_i), 2)
                d /= len(centroids)
                new_c = (x_i, d) if new_c[1] < d else new_c
        centroids.append(new_c[0])
    return np.matrix(centroids)    

In [ ]:
x, y, z = mat.shape

In [ ]:
pixels = [list(mat[i][j]) + [i, j] for i in range(len(mat)) for j in range(len(mat[i])) ]
pixels = pd.DataFrame(pixels, columns=['R', 'G', 'B', 'i', 'j'])
pixels

In [ ]:
class K_Means():
    def __init__(self, n, distance=euclidean, cc=rand_centroids):
        self.data = None
        self.n = n          # n° centroids
        self.K = None
        self.d = distance
        self.cc = cc        # centroid criteria

    def new_centroids(self, idx):
        new_K = [[] for _ in self.K]

        for i in range(len(self.K)):
            # obtener indices de todos los elementos pertenecientes al cluster i de la lista de indeices
            idx_curr_cluster = idx[idx['cluster'] == i]['idx'].to_list()
            # obtener elementos correspondientes al cluster i
            cluster_i_data = self.data.loc[idx_curr_cluster]
            
            if cluster_i_data.empty:
                new_K[i] = self.K[i]
            else:
                new_K[i] = self.data.loc[idx_curr_cluster].mean().to_numpy()
            
        return np.array(new_K, dtype=object)

    def label(self):
        """
        for each row of DataFrame 
            generate 3-tuple for each cluster
                [(row_id, cluster_i, distance), ...]
            get 2-tuple with the minimun distance
                (row_id, cluster_i)
        trasform list [(row_id, cluster_i), ...] to DataFrame and return it
        """
        idx = [
                min([(idx, i, self.d(self.K[i], self.data.iloc[idx].to_numpy())) for i in range(len(self.K))], key=itemgetter(2))[:-1] 
                for idx in range(len(self.data))
            ]
        return pd.DataFrame(idx, columns=['idx', 'cluster'])  

    def execute(self, d):
        self.data = d
        new_K = self.cc(self.data, self.n, self.d)

        i = 0
        idx = None
        while not (self.K == new_K).all():
            self.K = new_K
            idx = self.label()
            new_K = self.new_centroids(idx)
            print(new_K)
            time.sleep(5)
            i += 1
        print(f'\titerations {i}')
        return new_K, idx

In [ ]:
K = 4
data = pixels.copy()
data.loc[:, ['R', 'G', 'B']]

In [ ]:
k = K_Means(K, cc=plus_plus)

In [ ]:
kernels, clusters = k.execute(data)